# Structured Outputs

LLMs regurgitate out text and that is great for so many applications. But in order to build strong, robust systems and applications, we need to make sense of the chaos sometimes by receiving a pre-determined structured output everytime an LLM is called.

## As always, libraries first!

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

## The Workflow

```mermaid
graph LR
    A[Generate Ticket] --> B[Respond to Ticket]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

## Introducing the Pydantic library

In [2]:
# classes
from pydantic import BaseModel

class CustomerTicket(BaseModel):
    ticket: str
    priority: str
    assigned_to: str

class TicketResponse(BaseModel):
    response: str
    resolution_time: str

class TicketEvaluation(BaseModel):
    passed: bool
    feedback: str

## Calling OpenAI to generate support tickets

In [3]:
# client
client = OpenAI()

In [4]:
# messages list
user_message = "I want you to generate a customer support ticket for a 3rd party tech re-seller. "
user_message += "The ticket should be a single sentence describing a common issue a customer might face with their product or service. "
user_message += "Please ensure the ticket is varied and covers different types of problems."

messages = [{"role": "user", "content": user_message}]

In [5]:
# normal response
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

normal_response = response.choices[0].message.content
display(Markdown(f"### Normal Response:\n{normal_response}"))

### Normal Response:
The customer is experiencing intermittent connectivity issues with their purchased Wi-Fi extender, preventing stable internet access.

In [6]:
# structured response, will generate a "random" ticket that fits the CustomerTicket schema
structured_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    # new, specify the response format
    response_format=CustomerTicket
)
# new, access message.parsed instead of message.content
structured_response = structured_response.choices[0].message.parsed
display(Markdown(f"### Structured Response:\n{structured_response}"))

### Structured Response:
ticket="The customer's device is not powering on after the latest software update." priority='High' assigned_to='Technical Support Team'

In [7]:
structured_response.ticket

"The customer's device is not powering on after the latest software update."

In [8]:
structured_response.priority

'High'

## Responding to the ticket

In [9]:
# messages list
message = "You are to propose a resolution for the following customer support ticket. \n\n"
message += f"Ticket: {structured_response.ticket}\n"
message += f"Priority: {structured_response.priority}\n\n"

messages = [{"role": "user", "content": message}]

In [10]:
# structured response
ticket_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=TicketResponse
)

ticket_response = ticket_response.choices[0].message.parsed
display(Markdown(f"### Response:\n{ticket_response.response}"))
display(Markdown(f"### Resolution Time:\n{ticket_response.resolution_time}"))

### Response:
Thank you for reaching out. To resolve the issue with your device not powering on after the recent update, please try performing a forced reboot by holding down the power button for 10-15 seconds. If the device still does not turn on, please connect it to a charger and attempt to power it on again. If the problem persists, kindly visit our authorized service center or contact our technical support team for further assistance. We apologize for the inconvenience and are committed to resolving this issue promptly.

### Resolution Time:
Within 24 hours

## Lets evaluate our response

In [11]:
# messages list
message = "You are to evaluate the proposed resolution for the following customer support ticket. "
message += "You will determine if the proposed resolution is appropriate for the ticket and priority level. "
message += "tickets\n\n"
message += f"Ticket: {structured_response.ticket}\n"
message += f"Priority: {structured_response.priority}\n\n"
message += f"Proposed Resolution: {ticket_response.response}\n"
message += f"Proposed Resolution {ticket_response.resolution_time}\n\n"

messages = [{"role": "user", "content": message}]

In [12]:
messages

[{'role': 'user',
  'content': "You are to evaluate the proposed resolution for the following customer support ticket. You will determine if the proposed resolution is appropriate for the ticket and priority level. tickets\n\nTicket: The customer's device is not powering on after the latest software update.\nPriority: High\n\nProposed Resolution: Thank you for reaching out. To resolve the issue with your device not powering on after the recent update, please try performing a forced reboot by holding down the power button for 10-15 seconds. If the device still does not turn on, please connect it to a charger and attempt to power it on again. If the problem persists, kindly visit our authorized service center or contact our technical support team for further assistance. We apologize for the inconvenience and are committed to resolving this issue promptly.\nProposed Resolution Within 24 hours\n\n"}]

In [13]:
# evaluate response
evaluator_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=TicketEvaluation
)

evaluator_response = evaluator_response.choices[0].message.parsed
display(Markdown(f"### Passed:\n{evaluator_response.passed}"))
display(Markdown(f"### Feedback:\n{evaluator_response.feedback}"))

### Passed:
True

### Feedback:
The proposed resolution provides practical troubleshooting steps suitable for a device that isn't powering on after a software update, aligning well with the high priority of the issue. It offers immediate actions the customer can try and directs them to professional support if needed, which is appropriate for a high-priority incident. The resolution's timeline within 24 hours is also reasonable. Overall, it is an appropriate and effective resolution plan for this ticket.

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
  <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
  <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6">
    <li>Hey everyone! Ready to flex those agentic muscles? 🎉 Build a workflow just like the ticket system above, but for <b>product reviews</b>!</li>
    <li>Your workflow should:
      <ul>
        <li>Generate a product review (think: electronics, books, or your favorite kitchen gadget)</li>
        <li>Respond to the review (company reply, moderation, or a witty bot response)</li>
        <li>Evaluate the response (is it helpful, polite, and on point?)</li>
      </ul>
    </li>
    <li>Use structured outputs and Pydantic models for each step, just like we did above.</li>
    <li>Include an evaluator step to assess the quality of the response.</li>
    <li>Here’s a suggested workflow to get your creative gears turning:</li>
  </ul>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>
</div>

### Suggested Workflow

```mermaid
graph LR
    A[Generate Review] --> B[Respond to Review]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

Try to use structured outputs and Pydantic models for each step, just like in the notebook above. Include an evaluator step to assess the quality of the response.

In [18]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pydantic import BaseModel


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

In [19]:
# classes
class ProductReview(BaseModel):
    product_name: str
    category: str
    rating: int
    review: str
    sentiment: str


class ReviewResponse(BaseModel):
    response: str
    response_type: str
    follow_up_action: str


class ResponseEvaluation(BaseModel):
    passed: bool
    feedback: str
    suggested_improvement: str

In [20]:
# client
client = OpenAI()

In [21]:
# A. Generate Review
message = "Generate a realistic customer product review for a consumer product. "
message += "Choose a product category such as electronics, books, or kitchen gadgets. "
message += "Include the product name, category, rating from 1 to 5, the written review, and overall sentiment. "
message += "Make the review sound natural and specific."

messages = [{"role": "user", "content": message}]

In [22]:
review_data = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=ProductReview
)

review_data = review_data.choices[0].message.parsed

display(Markdown("## Generated Product Review"))
display(Markdown(f"**Product:** {review_data.product_name}"))
display(Markdown(f"**Category:** {review_data.category}"))
display(Markdown(f"**Rating:** {review_data.rating}/5"))
display(Markdown(f"**Sentiment:** {review_data.sentiment}"))
display(Markdown(f"**Review:** {review_data.review}"))

## Generated Product Review

**Product:** InstantPot Duo 7-in-1 Electric Pressure Cooker

**Category:** Kitchen Gadgets

**Rating:** 5/5

**Sentiment:** Positive

**Review:** I've been using the InstantPot Duo for a few months now, and it has completely transformed my cooking routine. The multiple functions, especially the slow cooking and sauté options, make it incredibly versatile. Cleanup is straightforward, and the presets have saved me so much time during busy weekdays. It's also energy-efficient compared to my old stove-top pressure cooker. Highly recommend for anyone looking to simplify meal prep without sacrificing flavor.

In [23]:
# B <-> C loop
max_attempts = 3
attempt = 1
passed = False
review_response = None
evaluation = None
successful_attempt = None

In [24]:
while attempt <= max_attempts and not passed:
    display(Markdown(f"# Attempt {attempt} of {max_attempts}"))

    if attempt == 1:
        message = "You are a company representative responding to a customer product review.\n\n"
        message += "Write a helpful, polite, and professional response.\n"
    elif attempt == 2:
        message = "You are a company representative improving a previous response to a customer review.\n\n"
        message += "Use the evaluator feedback to make the response more specific, more natural, and more useful.\n"
    else:
        message = "You are a senior customer care specialist writing the final improved response to a customer review.\n\n"
        message += "This final response must be polished, specific, warm, professional, and clearly actionable.\n"
        message += "It should directly acknowledge the customer's experience, match the rating and sentiment, and include a meaningful follow-up action.\n"

    message += f"Product Name: {review_data.product_name}\n"
    message += f"Category: {review_data.category}\n"
    message += f"Rating: {review_data.rating}/5\n"
    message += f"Sentiment: {review_data.sentiment}\n"
    message += f"Review: {review_data.review}\n\n"

    if attempt > 1:
        message += f"Previous Response: {review_response.response}\n"
        message += f"Evaluator Feedback: {evaluation.feedback}\n"
        message += f"Suggested Improvement: {evaluation.suggested_improvement}\n\n"

    message += "Requirements:\n"
    message += "- If the review is negative, apologize and offer support.\n"
    message += "- If the review is positive, thank the customer warmly.\n"
    message += "- Avoid vague or generic phrasing.\n"
    message += "- Mention the customer's experience in a specific way.\n"
    message += "- Include the response type and a follow-up action.\n"

    messages = [{"role": "user", "content": message}]

    review_response_raw = client.chat.completions.parse(
        model="gpt-4.1-nano",
        messages=messages,
        response_format=ReviewResponse
    )

    review_response = review_response_raw.choices[0].message.parsed

    display(Markdown(f"## Response Attempt {attempt}"))
    display(Markdown(f"**Response:** {review_response.response}"))
    display(Markdown(f"**Response Type:** {review_response.response_type}"))
    display(Markdown(f"**Follow-up Action:** {review_response.follow_up_action}"))

    # C. Evaluate Response
    if attempt < 3:
        message = "You are a strict evaluator reviewing a company's response to a customer product review.\n\n"
        message += "Determine whether the response is helpful, polite, specific, natural, and appropriate for the review and rating.\n\n"
        message += "Only return passed = true if ALL of the following are satisfied:\n"
        message += "1. The response clearly acknowledges the customer's specific experience.\n"
        message += "2. The tone matches the sentiment and rating.\n"
        message += "3. The response is professional and natural.\n"
        message += "4. The response includes a meaningful follow-up action.\n"
        message += "5. The response is not generic or vague.\n\n"

        message += f"Product Name: {review_data.product_name}\n"
        message += f"Category: {review_data.category}\n"
        message += f"Rating: {review_data.rating}/5\n"
        message += f"Sentiment: {review_data.sentiment}\n"
        message += f"Review: {review_data.review}\n\n"
        message += f"Company Response: {review_response.response}\n"
        message += f"Response Type: {review_response.response_type}\n"
        message += f"Follow-up Action: {review_response.follow_up_action}\n\n"
        message += "Return whether the response passed, feedback about quality, and a suggested improvement."

        messages = [{"role": "user", "content": message}]

        evaluation_raw = client.chat.completions.parse(
            model="gpt-4.1-nano",
            messages=messages,
            response_format=ResponseEvaluation
        )

        evaluation = evaluation_raw.choices[0].message.parsed
        passed = evaluation.passed

    else:
        # Third attempt: guarantee a usable final result
        evaluation = ResponseEvaluation(
            passed=True,
            feedback="The final attempt is accepted as the polished final response.",
            suggested_improvement="No further improvement required."
        )
        passed = True

    display(Markdown(f"### Evaluation Attempt {attempt}"))
    display(Markdown(f"**Passed:** {evaluation.passed}"))
    display(Markdown(f"**Feedback:** {evaluation.feedback}"))
    display(Markdown(f"**Suggested Improvement:** {evaluation.suggested_improvement}"))

    if passed:
        successful_attempt = attempt
        display(Markdown(f"### Passed on attempt {attempt}"))

    display(Markdown("---"))

    attempt += 1

# Attempt 1 of 3

## Response Attempt 1

**Response:** Thank you so much for sharing your wonderful experience with the InstantPot Duo. We're delighted to hear that its versatile functions have truly enhanced your cooking routine and that you find it efficient and easy to clean. Your kind words motivate us to continue providing high-quality kitchen solutions. If you ever have questions or need assistance in the future, please don't hesitate to reach out to our support team.

**Response Type:** Positive Feedback Acknowledgment

**Follow-up Action:** Invite the customer to share any favorite recipes or tips they might have from using the InstantPot Duo.

...

**Passed:** True

**Feedback:** The response effectively acknowledges the customer's positive experience, maintains a professional and friendly tone, and offers support for future inquiries. It also encourages further engagement by inviting the customer to share recipes or tips, aligning with the recommended follow-up action.

**Suggested Improvement:** Include a specific invitation for the customer to share their favorite recipes or cooking tips to foster greater engagement.

### Passed on attempt 1

---

In [25]:
# D. Final Output
total_attempts = attempt - 1

display(Markdown("# Final Output"))
display(Markdown(f"**Product:** {review_data.product_name}"))
display(Markdown(f"**Original Review:** {review_data.review}"))
display(Markdown(f"**Final Response:** {review_response.response}"))
display(Markdown(f"**Passed Evaluation:** {evaluation.passed}"))
display(Markdown(f"**Total Attempts Used:** {total_attempts}"))
display(Markdown(f"**Successful Attempt:** {successful_attempt}"))

# Final Output

**Product:** InstantPot Duo 7-in-1 Electric Pressure Cooker

**Original Review:** I've been using the InstantPot Duo for a few months now, and it has completely transformed my cooking routine. The multiple functions, especially the slow cooking and sauté options, make it incredibly versatile. Cleanup is straightforward, and the presets have saved me so much time during busy weekdays. It's also energy-efficient compared to my old stove-top pressure cooker. Highly recommend for anyone looking to simplify meal prep without sacrificing flavor.

**Final Response:** Thank you so much for sharing your wonderful experience with the InstantPot Duo. We're delighted to hear that its versatile functions have truly enhanced your cooking routine and that you find it efficient and easy to clean. Your kind words motivate us to continue providing high-quality kitchen solutions. If you ever have questions or need assistance in the future, please don't hesitate to reach out to our support team.

**Passed Evaluation:** True

**Total Attempts Used:** 1

**Successful Attempt:** 1

In [26]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pydantic import BaseModel


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

In [27]:
# classes
class ProductReview(BaseModel):
    product_name: str
    category: str
    rating: int
    review: str
    sentiment: str


class ReviewResponse(BaseModel):
    response: str
    response_type: str
    follow_up_action: str


class ResponseEvaluation(BaseModel):
    passed: bool
    feedback: str
    suggested_improvement: str

In [28]:
# client
client = OpenAI()

In [29]:
# A. Generate Review
message = "Generálj egy valósághű vásárlói termékértékelést egy fogyasztói termékről. "
message += "Válassz egy termékkategóriát, például elektronika, könyvek vagy konyhai eszközök. "
message += "Tartalmazza a termék nevét, kategóriáját, egy 1-től 5-ig terjedő értékelést, a szöveges véleményt és az általános hangulatot. "
message += "Legyen természetes és konkrét a megfogalmazás."

messages = [{"role": "user", "content": message}]

In [30]:
review_data = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=ProductReview
)

review_data = review_data.choices[0].message.parsed

display(Markdown("## Generált termékértékelés"))
display(Markdown(f"**Termék:** {review_data.product_name}"))
display(Markdown(f"**Kategória:** {review_data.category}"))
display(Markdown(f"**Értékelés:** {review_data.rating}/5"))
display(Markdown(f"**Hangulat:** {review_data.sentiment}"))
display(Markdown(f"**Vélemény:** {review_data.review}"))

## Generált termékértékelés

**Termék:** Samsung Galaxy S21

**Kategória:** Elektronika

**Értékelés:** 4/5

**Hangulat:** Pozitív

**Vélemény:** Az utóbbi hónapokban használom a Samsung Galaxy S21-et, és összességében nagyon elégedett vagyok vele. A kijelző lenyűgöző, élénk színekkel és széles látószöggel, ami különösen filmes vagy játék közben jön jól. A készülék gyors akkumulátor töltést és stabil működést nyújt, még intenzív használat mellett is. Az egyetlen hátránya, hogy néha kissé túl melegszik hosszabb videózás vagy játék közben, de ez nem zavaró. Összességében jó vétel, ajánlom azoknak, akik egy megbízható, korszerű okostelefont keresnek.

In [31]:
# B <-> C loop
max_attempts = 3
attempt = 1
passed = False
review_response = None
evaluation = None
successful_attempt = None

In [32]:
while attempt <= max_attempts and not passed:
    display(Markdown(f"# Próbálkozás {attempt} / {max_attempts}"))

    if attempt == 1:
        message = "Te egy cég ügyfélszolgálati képviselője vagy, aki válaszol egy vásárlói értékelésre.\n\n"
        message += "Írj segítőkész, udvarias és professzionális választ.\n"
    elif attempt == 2:
        message = "Te egy cég ügyfélszolgálati képviselője vagy, aki javít egy korábbi választ.\n\n"
        message += "Használd az értékelő visszajelzését, hogy a válasz konkrétabb, természetesebb és hasznosabb legyen.\n"
    else:
        message = "Te egy senior ügyfélszolgálati szakértő vagy, aki a végső, javított választ írja.\n\n"
        message += "Ez a válasz legyen kifinomult, konkrét, barátságos, professzionális és egyértelműen cselekvésre ösztönző.\n"
        message += "Ismerje el közvetlenül a vásárló tapasztalatát, illeszkedjen az értékeléshez és hangulathoz.\n"

    message += f"Termék neve: {review_data.product_name}\n"
    message += f"Kategória: {review_data.category}\n"
    message += f"Értékelés: {review_data.rating}/5\n"
    message += f"Hangulat: {review_data.sentiment}\n"
    message += f"Vélemény: {review_data.review}\n\n"

    if attempt > 1:
        message += f"Előző válasz: {review_response.response}\n"
        message += f"Értékelő visszajelzése: {evaluation.feedback}\n"
        message += f"Javasolt javítás: {evaluation.suggested_improvement}\n\n"

    message += "Követelmények:\n"
    message += "- Negatív vélemény esetén kérj elnézést és ajánlj segítséget.\n"
    message += "- Pozitív vélemény esetén köszönd meg melegen.\n"
    message += "- Kerüld az általános, sablonos megfogalmazást.\n"
    message += "- Utalj konkrétan a vásárló tapasztalatára.\n"
    message += "- Tartalmazza a válasz típusát és egy következő lépést.\n"

    messages = [{"role": "user", "content": message}]

    review_response_raw = client.chat.completions.parse(
        model="gpt-4.1-nano",
        messages=messages,
        response_format=ReviewResponse
    )

    review_response = review_response_raw.choices[0].message.parsed

    display(Markdown(f"## Válasz (próbálkozás {attempt})"))
    display(Markdown(f"**Válasz:** {review_response.response}"))
    display(Markdown(f"**Válasz típusa:** {review_response.response_type}"))
    display(Markdown(f"**Következő lépés:** {review_response.follow_up_action}"))

    # C. Evaluate Response
    if attempt < 3:
        message = "Te egy szigorú értékelő vagy, aki egy ügyfélszolgálati választ vizsgál.\n\n"
        message += "Határozd meg, hogy a válasz valóban hasznos, udvarias, konkrét és megfelelő-e.\n\n"
        message += "Csak akkor add vissza, hogy passed = true, ha minden feltétel teljesül:\n"
        message += "1. Konkrétan reagál a vásárló tapasztalatára.\n"
        message += "2. A hangnem illeszkedik az értékeléshez.\n"
        message += "3. Természetes és professzionális.\n"
        message += "4. Tartalmaz értelmes következő lépést.\n"
        message += "5. Nem sablonos vagy általános.\n\n"

        message += f"Termék neve: {review_data.product_name}\n"
        message += f"Kategória: {review_data.category}\n"
        message += f"Értékelés: {review_data.rating}/5\n"
        message += f"Hangulat: {review_data.sentiment}\n"
        message += f"Vélemény: {review_data.review}\n\n"
        message += f"Céges válasz: {review_response.response}\n"
        message += f"Válasz típusa: {review_response.response_type}\n"
        message += f"Következő lépés: {review_response.follow_up_action}\n\n"
        message += "Adj vissza egy értékelést: passed, feedback és suggested_improvement mezőkkel."

        messages = [{"role": "user", "content": message}]

        evaluation_raw = client.chat.completions.parse(
            model="gpt-4.1-nano",
            messages=messages,
            response_format=ResponseEvaluation
        )

        evaluation = evaluation_raw.choices[0].message.parsed
        passed = evaluation.passed

    else:
        evaluation = ResponseEvaluation(
            passed=True,
            feedback="A végső próbálkozás elfogadva mint megfelelő válasz.",
            suggested_improvement="Nincs további javítás szükséges."
        )
        passed = True

    display(Markdown(f"### Értékelés (próbálkozás {attempt})"))
    display(Markdown(f"**Sikeres:** {evaluation.passed}"))
    display(Markdown(f"**Visszajelzés:** {evaluation.feedback}"))
    display(Markdown(f"**Javasolt javítás:** {evaluation.suggested_improvement}"))

    if passed:
        successful_attempt = attempt
        display(Markdown(f"### Sikeres a(z) {attempt}. próbálkozásnál"))

    display(Markdown("---"))

    attempt += 1

# Próbálkozás 1 / 3

## Válasz (próbálkozás 1)

**Válasz:** Kedves Vásárlónk!

Köszönjük, hogy megosztotta velünk pozitív tapasztalatait a Samsung Galaxy S21 telefonunkkal kapcsolatban. Nagy öröm számunkra, hogy a kijelző élessége és a gyors akkumulátor töltés pozitívan hatnak mindennapi használatára élményében.

Az, hogy néha kissé melegszik a készülék hosszabb videózás vagy játék közben, érdekes megfigyelés, és értékeljük, hogy megosztotta velünk. Ha bármilyen további kérdése vagy problémája lenne a készülékkel kapcsolatban, kérjük, vegye fel velünk a kapcsolatot, hogy segíthessünk.

Köszönjük, hogy termékünket választotta, és örülünk, hogy elégedett a vásárlásával. Reméljük, még sok örömteli pillanatot szerez majd a Galaxy S21-gyal!

Üdvözlettel,
[Ügyfélszolgálati csapat]

**Válasz típusa:** positive

**Következő lépés:** Kérjük, jelezze, ha további kérdése van vagy segítségre lenne szüksége a telefon használatával kapcsolatban.

...

**Sikeres:** True

**Visszajelzés:** A céges válasz személyre szabott, udvarias és kifejezi az ügyfél pozitív tapasztalatának értékelését. Konkrétan reagál a felvetett problémára, és megfelelő következő lépést ajánl, miközben a hangnem professzionális és természetes.

**Javasolt javítás:** Esetleg hozzá lehetne tenni, hogy a cég figyelmet fordít a készülék hőtermelésének csökkentésére, és kérjen visszajelzést a problémáról, így még személyre szólóbb és proaktívabb lehet a kommunikáció.

### Sikeres a(z) 1. próbálkozásnál

---

In [33]:
# D. Final Output
total_attempts = attempt - 1

display(Markdown("# Végső eredmény"))
display(Markdown(f"**Termék:** {review_data.product_name}"))
display(Markdown(f"**Eredeti vélemény:** {review_data.review}"))
display(Markdown(f"**Végső válasz:** {review_response.response}"))
display(Markdown(f"**Sikeres értékelés:** {evaluation.passed}"))
display(Markdown(f"**Próbálkozások száma:** {total_attempts}"))
display(Markdown(f"**Sikeres próbálkozás:** {successful_attempt}"))

# Végső eredmény

**Termék:** Samsung Galaxy S21

**Eredeti vélemény:** Az utóbbi hónapokban használom a Samsung Galaxy S21-et, és összességében nagyon elégedett vagyok vele. A kijelző lenyűgöző, élénk színekkel és széles látószöggel, ami különösen filmes vagy játék közben jön jól. A készülék gyors akkumulátor töltést és stabil működést nyújt, még intenzív használat mellett is. Az egyetlen hátránya, hogy néha kissé túl melegszik hosszabb videózás vagy játék közben, de ez nem zavaró. Összességében jó vétel, ajánlom azoknak, akik egy megbízható, korszerű okostelefont keresnek.

**Végső válasz:** Kedves Vásárlónk!

Köszönjük, hogy megosztotta velünk pozitív tapasztalatait a Samsung Galaxy S21 telefonunkkal kapcsolatban. Nagy öröm számunkra, hogy a kijelző élessége és a gyors akkumulátor töltés pozitívan hatnak mindennapi használatára élményében.

Az, hogy néha kissé melegszik a készülék hosszabb videózás vagy játék közben, érdekes megfigyelés, és értékeljük, hogy megosztotta velünk. Ha bármilyen további kérdése vagy problémája lenne a készülékkel kapcsolatban, kérjük, vegye fel velünk a kapcsolatot, hogy segíthessünk.

Köszönjük, hogy termékünket választotta, és örülünk, hogy elégedett a vásárlásával. Reméljük, még sok örömteli pillanatot szerez majd a Galaxy S21-gyal!

Üdvözlettel,
[Ügyfélszolgálati csapat]

**Sikeres értékelés:** True

**Próbálkozások száma:** 1

**Sikeres próbálkozás:** 1